In [ ]:
import sys
import math

# Please do not remove package declarations because these are used by the autograder.

import sys
import math
from collections import defaultdict
from math import log


def legal_next(s,n_match):
        if s in ('S', 'I0'):
            return ['I0', 'E'] if n_match == 0 else ['I0', 'M1', 'D1']
        if s == 'E':
            return []
        kind, k = s[0], int(s[1:])
        if kind in ('M', 'D'):
            return [f'I{k}', 'E'] if k == n_match else [f'I{k}', f'M{k+1}', f'D{k+1}']
        # I state
        return [f'I{k}', 'E'] if k == n_match else [f'I{k}', f'M{k+1}', f'D{k+1}']

def pseudo_trans(counts, opts,sigma):
        total = sum(counts.get(o, 0) for o in opts)
        n = len(opts)
        return [(((counts.get(o, 0) / total) if total > 0 else 1.0 / n) + sigma)
                / (1 + n * sigma)
                for o in opts]

def pseudo_emit(counts,alphabet,sigma,n_alph):
    total = sum(counts.get(a, 0) for a in alphabet)
    return [(((counts.get(a, 0) / total) if total > 0 else 1.0 / n_alph) + sigma)
            / (1 + n_alph * sigma)
            for a in alphabet]

def align_sequence_to_profile_hmm(x: str,
                                  theta: float,
                                  sigma: float,
                                  alphabet: list[str],
                                  alignment: list[str]) -> list[str]:
    """
    Align x to HMM(Alignment, theta, sigma) and return the optimal hidden path
    as a list of state names.
    """

    
    n_seq, n_col, n_alph = len(alignment), len(alignment[0]), len(alphabet)

    is_match = [sum(seq[c] == '-' for seq in alignment) / n_seq < theta
                for c in range(n_col)]
    n_match  = sum(is_match)

    states = ['S', 'I0']
    for i in range(1, n_match + 1):
        states += [f'M{i}', f'D{i}', f'I{i}']
    states.append('E')
    sidx = {s: i for i, s in enumerate(states)}
    K    = len(states)

    

    # Count transitions and emissions from alignment
    trans_counts = defaultdict(lambda: defaultdict(float))
    emit_counts  = defaultdict(lambda: defaultdict(float))

    for seq in alignment:
        path, current = ['S'], 0
        for c in range(n_col):
            ch = seq[c]
            if is_match[c]:
                current += 1
                path.append(f'D{current}' if ch == '-' else f'M{current}')
            elif ch != '-':
                path.append(f'I{current}')
        path.append('E')

        for i in range(len(path) - 1):
            trans_counts[path[i]][path[i+1]] += 1

        col = 0
        for st in path[1:-1]:
            if st.startswith('M'):
                while not is_match[col]: col += 1
                emit_counts[st][seq[col]] += 1
                col += 1
            elif st.startswith('D'):
                while not is_match[col]: col += 1
                col += 1
            else:   # I
                while col < n_col and is_match[col]: col += 1
                emit_counts[st][seq[col]] += 1
                col += 1

    # p_new = (p_raw + sigma) / (1 + n_options * sigma)
    

    T = [[0.0] * K     for _ in range(K)]
    E = [[0.0] * n_alph for _ in range(K)]

    for s in states:
        si   = sidx[s]
        opts = legal_next(s,n_match)
        if not opts:
            continue
        for o, p in zip(opts, pseudo_trans(trans_counts[s], opts,sigma)):
            T[si][sidx[o]] = p
        if s[0] in ('M', 'I'):
            for j, p in enumerate(pseudo_emit(emit_counts[s],alphabet,sigma,n_alph)):
                E[si][j] = p

    # ── Viterbi with silent-state handling 
    # dp[i][k] = best log-prob having consumed exactly i chars of x in state k.
    # Silent states (S, D*, E) stay at the same i; emitting states (M*, I*) advance to i+1.
    # traceback from E at level n so every valid path terminates properly.

    NEG_INF = float('-inf')
    n       = len(x)
    sym     = {c: i for i, c in enumerate(alphabet)}

    def silent(s): return s == 'S' or s == 'E' or s.startswith('D')

    dp  = [[NEG_INF] * K for _ in range(n + 1)]
    ptr = [[None]    * K for _ in range(n + 1)]
    dp[0][sidx['S']] = 0.0

    for i in range(n + 1):
        
        changed = True
        while changed:
            changed = False
            for k, s in enumerate(states):
                if dp[i][k] == NEG_INF:
                    continue
                for j, t in enumerate(T[k]):
                    if t <= 0 or not silent(states[j]):
                        continue
                    v = dp[i][k] + log(t)
                    if v > dp[i][j]:
                        dp[i][j]  = v
                        ptr[i][j] = (i, k)
                        changed   = True

        if i == n:
            break

        
        si = sym[x[i]]
        for k, s in enumerate(states):
            if dp[i][k] == NEG_INF:
                continue
            for j, t in enumerate(T[k]):
                if t <= 0 or silent(states[j]):
                    continue
                em = E[j][si]
                if em <= 0:
                    continue
                v = dp[i][k] + log(t) + log(em)
                if v > dp[i + 1][j]:
                    dp[i + 1][j]  = v
                    ptr[i + 1][j] = (i, k)

    # Traceback from E at level n
    path, cur = [], (n, sidx['E'])
    while cur is not None:
        i2, k = cur
        path.append(states[k])
        cur = ptr[i2][k]
    path.reverse()

    return [s for s in path if s not in ('S', 'E')]



In [ ]:

# Sequence Alignment with Profile HMM Problem: Align a new sequence to a family of sequences using a profile HMM.

# Input: A multiple alignment Alignment, a threshold θ, a pseudocount value σ, and a string Text.
# Output: An optimal hidden path emitting Text in HMM(Alignment, θ, σ).
# Code Challenge: Solve the Sequence Alignment with Profile HMM Problem.

# Input: A string x followed by a threshold θ and a pseudocount σ, followed by an alphabet Σ, followed by a multiple alignment Alignment whose strings are formed from Σ.
# Output: An optimal hidden path emitting x in HMM(Alignment, θ, σ).
# Debug Datasets

# Sample Input 1:

# AEFDFDC
# --------
# 0.4 0.01
# --------
# A B C D E F
# --------
# ACDEFACADF
# AFDA---CCF
# A--EFD-FDC
# ACAEF--A-C
# ADDEFAAADF
# Sample Output 1:

# M1 D2 D3 M4 M5 I5 M6 M7 M8
# Sample Input 2:

# AB
# --------
# 0.4 0.01
# --------
# A B
# --------
# AA
# AA
# Sample Output 2:

# M1 M2
# Sample Input 3:

# AB
# --------
# 0.4 0.01
# --------
# A B C D E
# --------
# AA
# AA
# Sample Output 3:

# M1 M2
# Sample Input 4:

# AB
# --------
# 0.4 0.01
# --------
# A B
# --------
# A-
# A-
# Sample Output 4:

# M1 I1
# Sample Input 5:

# AAAAAB
# --------
# 0.4 0.01
# --------
# A B
# --------
# -B
# -B
# Sample Output 5:

# I0 I0 I0 I0 I0 M1
# Sample Input 6:

# AB
# --------
# 0.4 0.01
# --------
# A B
# --------
# AAAAB
# AAAAB
# Sample Output 6:

# D1 D2 D3 M4 M5
# Sample Input 7:

# B
# --------
# 0.4 0.01
# --------
# A B
# --------
# AAAB
# AAAB
# ---B
# Sample Output 7:

# D1 D2 D3 M4